In [ ]:
### Deconvolution of 3peaks with BG correction

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit

# ==========================================
# 1. DATA LOADING & PRE-PROCESSING
# ==========================================
file_path = 'spectra_NiP_HER_1/1175vr.txt'
df = pd.read_csv(file_path, sep='\s+', comment='#', header=None, encoding='latin1')

range_min, range_max = 2500, 4000
df = df[(df[0] >= range_min) & (df[0] <= range_max)]
wn = df[0].values
inty_raw = df[1].values

# Smooth the raw data
inty_s = savgol_filter(inty_raw, window_length=31, polyorder=3)

# Background Correction using Anchors
# We define specific zones where we are sure no peaks exist
mask_anchors = ((wn >= 2500) & (wn <= 2550)) | ((wn >= 3950) & (wn <= 4000))
wn_anchors = wn[mask_anchors]
inty_anchors = inty_s[mask_anchors]

# Fit a 2nd order polynomial to the baseline "floor"
poly_coeffs = np.polyfit(wn_anchors, inty_anchors, 2)
poly_bg = np.polyval(poly_coeffs, wn)

# Subtract background and ensure baseline is at zero
inty_corrected = inty_s - poly_bg
inty_corrected -= inty_corrected.min()

# ==========================================
# 2. GAUSSIAN MODELS
# ==========================================
def gaussian(x, amp, ctr, sigma):
    return amp * np.exp(-(x - ctr)**2 / (2 * sigma**2))

def fit_model(x, a1, c1, s1, a2, c2, s2, a3, c3, s3):
    return (gaussian(x, a1, c1, s1) + 
            gaussian(x, a2, c2, s2) + 
            gaussian(x, a3, c3, s3))

# Initial Guesses & Bounds
max_h = max(inty_corrected)
guesses = [max_h*0.7, 3260, 15, max_h, 3434, 15, max_h*0.5, 3617, 15]
lower_b = [0, 3230, 5,  0, 3404, 5,  0, 3587, 5]
upper_b = [np.inf, 3290, 60, np.inf, 3464, 60, np.inf, 3647, 60]

popt, _ = curve_fit(fit_model, wn, inty_corrected, p0=guesses, bounds=(lower_b, upper_b))

# ==========================================
# 3. COMPREHENSIVE PLOTTING
# ==========================================
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, 
                               gridspec_kw={'height_ratios': [1, 2]})

# --- TOP SUBPLOT: Background Justification ---
ax1.plot(wn, inty_s, color='gray', lw=1, label='Smoothed Raw Data')
ax1.plot(wn, poly_bg, 'r--', lw=2, label='Fitted Background (Anchor-Fit)')
ax1.scatter(wn_anchors, inty_anchors, color='red', s=15, zorder=5, label='Anchor Points')
ax1.set_title("Step 1: Background Modeling & Anchor Identification", fontsize=12)
ax1.set_ylabel("Raw Intensity")
ax1.legend(loc='upper right', fontsize=9)

# --- BOTTOM SUBPLOT: Deconvolution ---
ax2.plot(wn, inty_corrected, 'k', lw=1.5, label='Corrected Data (Baseline=0)')
ax2.plot(wn, fit_model(wn, *popt), 'r:', lw=2, label='Cumulative Fit')

colors = ['#3498db', '#2ecc71', '#e67e22']
for i in range(3):
    idx = i * 3
    peak_y = gaussian(wn, *popt[idx:idx+3])
    ax2.fill_between(wn, 0, peak_y, color=colors[i], alpha=0.4, 
                     label=f'Peak {i+1}: {popt[idx+1]:.1f} $cm^{{-1}}$')

ax2.set_title("Step 2: 3-Peak Gaussian Deconvolution", fontsize=12)
ax2.set_xlabel("Wavenumber ($cm^{-1}$)", fontsize=12)
ax2.set_ylabel("Corrected Intensity")
ax2.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

# Print Results for Documentation
print(f"{'Peak':<8} | {'Center':<8} | {'FWHM':<8} | {'Area':<8}")
for i in range(3):
    a, c, s = popt[i*3 : i*3+3]
    fwhm = 2.355 * s
    area = a * s * np.sqrt(2 * np.pi)
    print(f"{i+1:<8} | {c:<8.1f} | {fwhm:<8.1f} | {area:<8.1f}")

In [ ]:
### Three peak deconvolution using Gaussian

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit

# ==========================================
# 1. DATA LOADING & PRE-PROCESSING
# ==========================================
file_path = 'spectra_Anod-SS1/a0525vrr.txt'
df = pd.read_csv(file_path, sep='\s+', comment='#', header=None, encoding='latin1')

# Define your spectral window
range_min, range_max = 2500, 4000
df = df[(df[0] >= range_min) & (df[0] <= range_max)]
wn = df[0].values
inty_raw = df[1].values

# Smoothing: Essential for interface studies to remove electronic noise
inty_s = savgol_filter(inty_raw, window_length=31, polyorder=3)

# Background Correction using Anchors (First/Last 40 units)
mask_anchors = ((wn >= 2500) & (wn <= 2550)) | ((wn >= 3950) & (wn <= 4000))
poly_coeffs = np.polyfit(wn[mask_anchors], inty_s[mask_anchors], 2)
poly_bg = np.polyval(poly_coeffs, wn)

inty_corrected = inty_s - poly_bg
inty_corrected -= inty_corrected.min() # Zero the baseline

# ==========================================
# 2. GAUSSIAN MODELS & MATH
# ==========================================
def gaussian(x, amp, ctr, sigma):
    """Standard Gaussian profile for inhomogeneous broadening."""
    return amp * np.exp(-(x - ctr)**2 / (2 * sigma**2))

def fit_model(x, a1, c1, s1, a2, c2, s2, a3, c3, s3):
    """Sum of three Gaussians."""
    return (gaussian(x, a1, c1, s1) + 
            gaussian(x, a2, c2, s2) + 
            gaussian(x, a3, c3, s3))

# Initial Guesses [amp, center, sigma]
max_h = max(inty_corrected)
guesses = [max_h*0.7, 3260, 15, 
           max_h, 3434, 15, 
           max_h*0.5, 3617, 15]

# Bounds: Prevents peaks from "swapping" or becoming negative
# Format: ([min_a1, min_c1, min_s1...], [max_a1, max_c1, max_s1...])
lower_bounds = [0, 3230, 5,  0, 3404, 5,  0, 3587, 5]
upper_bounds = [np.inf, 3290, 60, np.inf, 3464, 60, np.inf, 3647, 60]

popt, pcov = curve_fit(fit_model, wn, inty_corrected, p0=guesses, bounds=(lower_bounds, upper_bounds))

# ==========================================
# 3. ANALYSIS & PLOTTING
# ==========================================
plt.figure(figsize=(12, 7))

# Data Plotting
plt.plot(wn, inty_corrected, 'k', label='Experimental Data', lw=1.5, alpha=0.8)
total_fit = fit_model(wn, *popt)
plt.plot(wn, total_fit, 'r--', label='Cumulative Gaussian Fit', lw=2)

# Individual Peak Plotting
colors = ['#3498db', '#2ecc71', '#e67e22'] # Blue, Green, Orange
peak_labels = ["Peak A (e.g. $Fe_3O_4$)", "Peak B (e.g. $Cr_2O_3$)", "Peak C (e.g. $FeOOH$)"]

print(f"{'Peak':<10} | {'Center':<10} | {'FWHM':<10} | {'Area':<10}")
print("-" * 50)

for i in range(3):
    idx = i * 3
    # Extract params: amp, center, sigma
    a, c, s = popt[idx], popt[idx+1], popt[idx+2]
    
    # Gaussian Math: FWHM = 2 * sqrt(2 * ln(2)) * sigma ≈ 2.355 * sigma
    fwhm = 2.355 * s
    # Area = amp * sigma * sqrt(2 * pi)
    area = a * s * np.sqrt(2 * np.pi)
    
    # Plotting
    y_peak = gaussian(wn, a, c, s)
    plt.fill_between(wn, 0, y_peak, color=colors[i], alpha=0.4, label=f'{peak_labels[i]}')
    
    print(f"Peak {i+1:<5} | {c:<10.2f} | {fwhm:<10.2f} | {area:<10.2f}")

# Formatting
plt.title("3-Peak Gaussian Deconvolution: Electrolyte-Interface Analysis", fontsize=14)
plt.xlabel("Wavenumber ($cm^{-1}$)", fontsize=12)
plt.ylabel("Intensity (a.u.)", fontsize=12)
plt.legend(frameon=False)
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()